# Pipeline DoRA + INT4 de Gemma 4 E2B para Optimizacion de Prompts (AMD MI300X / ROCm 7.2.4)

Entorno objetivo: **AMD Instinct MI300X**, **ROCm 7.2.4**, **Ubuntu 24.04**. APIs actualizadas (sin argumentos deprecados).

Pasos del pipeline:

1. **Entorno**: verifica ROCm y la GPU.
2. **Descarga** de `google/gemma-4-E2B-it`.
3. **Entrenamiento con DoRA** (Weight-Decomposed LoRA) de **alto rango** sobre todas las proyecciones, con enfasis en **generalizar (no memorizar)**: early stopping, label smoothing, weight decay, lora_dropout, NEFTune. El adapter se **fusiona** en la base tras entrenar.
4. **Diagnostico** de memorizacion (brecha train vs eval).
5. **Cuantizacion INT4** (GPTQ, con fallback a NF4).
6. **Exportacion** a una carpeta autocontenida lista para Docker.

> Unico proposito del modelo: recibir un prompt, optimizarlo (menos tokens, sin redundancias, misma intencion) y devolver el prompt optimizado.

**Nota**: este pipeline NO construye la imagen Docker; solo genera la carpeta de export.

## Requisitos previos

- GPU **AMD Instinct MI300X** con **ROCm 7.2.4** en **Ubuntu 24.04** (o el contenedor oficial `rocm/pytorch`).
- Token de Hugging Face (licencia Gemma): `export HF_TOKEN=hf_...`.
- Tu **dataset** en `data/prompt_optimization.jsonl`, con los campos `input` (prompt original) y `output` (prompt optimizado) por linea.

Ejemplo de una linea (JSONL):
```json
{"input": "Por favor, si eres tan amable, me gustaria que me ayudaras a...", "output": "Ayudame a..."}
```

Para reducir memorizacion, el dataset debe ser **variado y no repetitivo**: muchos estilos de redundancia distintos, longitudes variadas y dominios diversos.

## 1. Verificacion del entorno ROCm / MI300X

In [ ]:
import subprocess, sys, platform

print('Python:', sys.version)
print('Plataforma:', platform.platform())

def sh(cmd):
    try:
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        return (out.stdout or '') + (out.stderr or '')
    except Exception as e:
        return '[no disponible] ' + str(e)

print('=== ROCm ===')
print(sh('cat /opt/rocm/.info/version 2>/dev/null || hipconfig --version 2>/dev/null'))
print('=== GPU (rocm-smi) ===')
print(sh('rocm-smi --showproductname --showmeminfo vram'))

try:
    import torch
    print('=== PyTorch ===')
    print('torch:', torch.__version__)
    print('HIP disponible:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Dispositivos:', torch.cuda.device_count())
        print('GPU 0:', torch.cuda.get_device_name(0))
        print('HIP version:', getattr(torch.version, 'hip', None))
        print('bf16 soportado:', torch.cuda.is_bf16_supported())
except ImportError:
    print('PyTorch aun no instalado; ejecuta la celda de instalacion.')

## 2. Instalacion de dependencias (ROCm 7.2)

PyTorch para ROCm 7.2 desde el indice oficial de pytorch.org. Alternativa probada por AMD: ruedas de `repo.radeon.com`. Si usas el contenedor `rocm/pytorch`, omite la instalacion de torch.

In [ ]:
%pip install --quiet --upgrade pip
# PyTorch para ROCm 7.2 (Ubuntu 24.04). Si ya estas en rocm/pytorch, comenta esta linea.
%pip install --quiet torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/rocm7.2
# Stack de entrenamiento actualizado (APIs no deprecadas).
%pip install --quiet --upgrade 'transformers>=4.57.0' 'trl>=0.21.0' 'datasets>=3.2.0' 'accelerate>=1.2.0' 'peft>=0.14.0' 'safetensors>=0.4.5' sentencepiece hf_transfer
# Cuantizacion INT4: gptqmodel (INT4 real, kernels ROCm) con fallback a bitsandbytes (NF4).
!pip install --quiet gptqmodel optimum || echo 'gptqmodel opcional no instalado'
!pip install --quiet bitsandbytes || echo 'bitsandbytes opcional no instalado'
print('Dependencias instaladas. Reinicia el kernel si actualizaste torch/transformers.')

## 3. Configuracion del pipeline

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    # --- Modelo base ---
    base_model_id: str = 'google/gemma-4-E2B-it'
    hf_token: str = ''  # o exporta HF_TOKEN en el entorno

    # --- Dataset (LO PROPORCIONAS TU) ---
    dataset_path: str = 'data/prompt_optimization.jsonl'
    prompt_field: str = 'input'         # campo con el prompt original
    optimized_field: str = 'output'     # campo con el prompt optimizado
    eval_ratio: float = 0.1
    max_seq_len: int = 2048

    # --- Entrenamiento DoRA (Weight-Decomposed LoRA de alto rango) ---
    output_dir: str = 'outputs/gemma4-e2b-prompt-optimizer-dora'      # adapter DoRA
    merged_dir: str = 'outputs/gemma4-e2b-prompt-optimizer-merged'    # base + adapter fusionado
    epochs: float = 3.0
    per_device_batch_size: int = 4
    grad_accum: int = 8
    learning_rate: float = 2e-4         # LR mas alto que en FFT (tipico en LoRA/DoRA)
    warmup_ratio: float = 0.05
    weight_decay: float = 0.05          # regularizacion L2 (anti-memorizacion)
    lr_scheduler: str = 'cosine'
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.1        # suaviza etiquetas -> menos memorizacion
    neftune_noise_alpha: float = 5.0    # ruido en embeddings -> mejor generalizacion
    early_stopping_patience: int = 3    # detiene si eval_loss deja de mejorar
    gradient_checkpointing: bool = True
    optim: str = 'adamw_torch_fused'
    logging_steps: int = 10
    eval_steps: int = 50
    save_steps: int = 50
    seed: int = 42

    # --- DoRA (alto rango -> gran capacidad de especializacion) ---
    lora_r: int = 128                   # rango alto (prueba 256 si tu VRAM lo permite)
    lora_alpha: int = 256               # normalmente 2x el rango
    lora_dropout: float = 0.05          # dropout del adapter (regularizacion)
    lora_target_modules: str = 'all-linear'  # todas las proyecciones (atencion + MLP)

    # --- Cuantizacion INT4 ---
    quant_out_dir: str = 'outputs/gemma4-e2b-prompt-optimizer-int4'
    quant_bits: int = 4
    quant_group_size: int = 128
    calib_samples: int = 256

    # --- Export ---
    export_dir: str = 'export/gemma4-e2b-prompt-optimizer'

cfg = Config()

INSTRUCTION = (
    'Optimiza el siguiente prompt: reescribelo con el minimo numero de tokens, '
    'sin redundancias ni relleno, conservando su intencion y significado. '
    'Devuelve UNICAMENTE el prompt optimizado, sin explicaciones.'
)

def build_messages(user_prompt, target=None):
    content = INSTRUCTION + '\n\n' + str(user_prompt).strip()
    msgs = [{'role': 'user', 'content': content}]
    if target is not None:
        msgs.append({'role': 'assistant', 'content': str(target).strip()})
    return msgs

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path('data').mkdir(parents=True, exist_ok=True)
print('Config lista. Modelo base:', cfg.base_model_id)

## 4. Carga y preparacion del dataset

Coloca tu dataset en `data/prompt_optimization.jsonl`. Si no existe, se crea un ejemplo minimo solo para probar el pipeline.

In [ ]:
import json, os, random
from datasets import Dataset, DatasetDict

def _read_records(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    records = []
    if path.endswith('.jsonl'):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    elif path.endswith('.json'):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        records = data if isinstance(data, list) else data.get('data', [])
    elif path.endswith('.csv'):
        import csv
        with open(path, 'r', encoding='utf-8') as f:
            records = list(csv.DictReader(f))
    else:
        raise ValueError('Formato no soportado: usa .jsonl, .json o .csv')
    return records

if not os.path.exists(cfg.dataset_path):
    print('AVISO: no se encontro', cfg.dataset_path, '- creando ejemplo minimo de demostracion.')
    demo = [
        {cfg.prompt_field: 'Por favor, me gustaria que si es posible me pudieras ayudar a redactar un correo muy formal para solicitar unas vacaciones.', cfg.optimized_field: 'Redacta un correo formal para solicitar vacaciones.'},
        {cfg.prompt_field: 'Quiero que hagas un resumen, pero que sea corto y conciso, del siguiente texto que te paso a continuacion.', cfg.optimized_field: 'Resume brevemente el siguiente texto.'},
    ]
    with open(cfg.dataset_path, 'w', encoding='utf-8') as f:
        for r in demo:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

records = _read_records(cfg.dataset_path)
print('Registros cargados:', len(records))

chat_records = [{'messages': build_messages(r[cfg.prompt_field], r[cfg.optimized_field])} for r in records]
random.seed(cfg.seed)
random.shuffle(chat_records)

n_eval = max(1, int(len(chat_records) * cfg.eval_ratio)) if len(chat_records) > 1 else 0
eval_records = chat_records[:n_eval]
train_records = chat_records[n_eval:]

ds = DatasetDict({
    'train': Dataset.from_list(train_records),
    'eval': Dataset.from_list(eval_records if eval_records else train_records[:1]),
})
print(ds)

## 5. Descarga del modelo base y del tokenizer

Se usa `dtype` (el antiguo `torch_dtype` esta deprecado). NO se inyecta `attention_dropout`: la arquitectura de Gemma no lo admite de forma dinamica; la regularizacion la aportan el `weight_decay`, el `lora_dropout` y NEFTune.

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = cfg.hf_token or os.environ.get('HF_TOKEN')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Flash-Attention 2 (kernels CK/Triton de ROCm) si esta disponible; si no, SDPA (siempre funciona en MI300X).
try:
    import flash_attn  # noqa: F401
    preferred_attn = 'flash_attention_2'
except Exception:
    preferred_attn = 'sdpa'

# 'dtype' reemplaza al deprecado 'torch_dtype'. (Sin attention_dropout: Gemma no lo admite dinamicamente.)
def _load_model(attn):
    kwargs = dict(token=hf_token, dtype=torch.bfloat16, attn_implementation=attn)
    try:
        return AutoModelForCausalLM.from_pretrained(cfg.base_model_id, **kwargs)
    except (ValueError, KeyError, OSError) as e:
        print('AutoModelForCausalLM fallo (', repr(e), '); probando loader multimodal texto-a-texto.')
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(cfg.base_model_id, **kwargs)

# Si FA2 falla al construir el modelo en tu ROCm, se reintenta con SDPA automaticamente.
try:
    model = _load_model(preferred_attn)
    attn_impl = preferred_attn
except Exception as e:
    print('Fallo con', preferred_attn, '(', repr(e), '); reintentando con SDPA.')
    model = _load_model('sdpa')
    attn_impl = 'sdpa'
print('attn_implementation =', attn_impl)

# use_cache debe estar desactivado durante el entrenamiento con gradient checkpointing.
model.config.use_cache = False

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print('Modelo cargado:', getattr(model.config, 'model_type', '?'), '| ~', round(n_params, 2), 'B params')

## 6. Entrenamiento con DoRA (Weight-Decomposed LoRA) de alto rango

En lugar de FFT usamos **DoRA**, que descompone la actualizacion de los pesos en **magnitud** y **direccion** (la direccion se adapta con LoRA de bajo rango y la magnitud con un parametro aprendible aparte). DoRA iguala o supera al FFT en tareas complejas manteniendo la red base **intacta y congelada**.

Para crear un **experto** con gran capacidad de aprendizaje:

- **Rango alto** (`r=128`, prueba `256`) y `lora_alpha=256`: cientos de millones de parametros entrenables.
- **`target_modules='all-linear'`**: se adaptan TODAS las proyecciones lineales (atencion Q/K/V/O + MLP gate/up/down), no solo la atencion.
- La base congelada evita el olvido catastrofico y reduce la memorizacion.

Tecnicas de regularizacion para que **aprenda y no memorice**:

- **Early stopping** sobre `eval_loss` + `load_best_model_at_end`.
- **Label smoothing**, **weight decay** y **lora_dropout**.
- **NEFTune** (ruido en embeddings) y **assistant_only_loss** (autodetectado).
- **bf16** + **gradient checkpointing**.

Tras entrenar, el adapter DoRA se **fusiona** en los pesos base (`merge_and_unload`), como recomienda PEFT para inferencia/cuantizacion, dejando un modelo completo listo para cuantizar a INT4.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from peft import LoraConfig

# Configuracion DoRA de alto rango sobre todas las proyecciones lineales.
peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.lora_target_modules,  # 'all-linear' -> atencion + MLP
    use_dora=True,                            # Weight-Decomposed LoRA
    bias='none',
    task_type='CAUSAL_LM',
)

# assistant_only_loss requiere que la plantilla de chat soporte etiquetas de generacion ({% generation %}).
# Si no las soporta, se entrena sobre toda la secuencia para evitar un error en tiempo de ejecucion.
_tmpl = tokenizer.chat_template or ''
use_assistant_only = 'generation' in _tmpl
if not use_assistant_only:
    print('AVISO: la plantilla de chat no soporta assistant_only_loss; se entrena sobre toda la secuencia.')

sft_config = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.per_device_batch_size,
    per_device_eval_batch_size=cfg.per_device_batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,
    label_smoothing_factor=cfg.label_smoothing,
    bf16=True,
    optim=cfg.optim,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    packing=False,
    max_length=cfg.max_seq_len,  # 'max_length' reemplaza al deprecado 'max_seq_length'
    neftune_noise_alpha=cfg.neftune_noise_alpha,
    assistant_only_loss=use_assistant_only,  # autodetectado segun la plantilla de chat
    logging_steps=cfg.logging_steps,
    eval_strategy='steps',       # 'eval_strategy' (no el deprecado 'evaluation_strategy')
    eval_steps=cfg.eval_steps,
    save_strategy='steps',
    save_steps=cfg.save_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=cfg.seed,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds['train'],
    eval_dataset=ds['eval'],
    processing_class=tokenizer,  # 'processing_class' reemplaza al deprecado 'tokenizer'
    peft_config=peft_config,     # TRL aplica DoRA (get_peft_model) automaticamente
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience)],
)

# Parametros entrenables reales (DoRA de alto rango -> cientos de millones).
trainer.model.print_trainable_parameters()

train_result = trainer.train()
trainer.save_model(cfg.output_dir)          # guarda el adapter DoRA
tokenizer.save_pretrained(cfg.output_dir)
print('Entrenamiento DoRA completado. Adapter guardado en', cfg.output_dir)
print(train_result.metrics)

# Fusionar el adapter DoRA en los pesos base (recomendado por PEFT para inferencia/cuantizacion).
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(cfg.merged_dir)
tokenizer.save_pretrained(cfg.merged_dir)
print('Modelo fusionado (base + DoRA) guardado en', cfg.merged_dir)

## 7. Diagnostico de memorizacion vs generalizacion

Una brecha grande entre `train_loss` y `eval_loss` indica memorizacion. Si es alta, amplia el dataset o sube la regularizacion (weight_decay / dropout / label_smoothing) y reduce epocas.

In [ ]:
log = trainer.state.log_history
tr = [x['loss'] for x in log if 'loss' in x and 'eval_loss' not in x]
ev = [x['eval_loss'] for x in log if 'eval_loss' in x]
if tr and ev:
    gap = ev[-1] - tr[-1]
    print('train_loss final:', round(tr[-1], 4))
    print('eval_loss  final:', round(ev[-1], 4))
    print('gap (eval - train):', round(gap, 4))
    if gap > 0.5:
        print('AVISO: brecha alta -> posible memorizacion. Sube weight_decay/dropout/label_smoothing, baja epochs o amplia el dataset.')
    else:
        print('OK: brecha moderada -> buena generalizacion.')
else:
    print('No hay suficientes puntos de evaluacion para el diagnostico.')

## 8. Prueba del modelo fusionado (base + DoRA, bf16)

In [ ]:
merged_model.config.use_cache = True

def optimize_prompt(m, tok, user_prompt, max_new_tokens=256):
    msgs = build_messages(user_prompt)
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(m.device)
    out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

tests = [
    'Hola, me preguntaba si por favor podrias ayudarme a redactar, de la forma mas profesional posible, una carta de presentacion para un puesto de ingeniero de software.',
    'Necesito que me expliques, de manera muy sencilla y facil de entender, como funciona paso a paso el algoritmo de ordenamiento quicksort.',
]
for t in tests:
    opt = optimize_prompt(merged_model, tokenizer, t)
    print('ORIGINAL   (', len(tokenizer(t)['input_ids']), 'tok):', t)
    print('OPTIMIZADO (', len(tokenizer(opt)['input_ids']), 'tok):', opt)
    print('-' * 80)

## 9. Cuantizacion INT4

Metodo principal: **GPTQ** con `gptqmodel` (INT4 real, con kernels optimizados para ROCm desde 6.2+). Si no esta disponible, **fallback a NF4 4-bit** con bitsandbytes. La calibracion usa tus propios prompts. Alternativa oficial de AMD: **Quark**.

In [ ]:
import gc, torch

# Se cuantiza el modelo YA FUSIONADO (base + DoRA), no el adapter suelto.
FULL_MODEL_DIR = cfg.merged_dir
QUANT_METHOD = None

# Liberamos memoria de los modelos en RAM/VRAM antes de cuantizar.
for _v in ('trainer', 'merged_model', 'model'):
    try:
        del globals()[_v]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()

# Datos de calibracion tomados de tus propios prompts.
calib_texts = [tokenizer.apply_chat_template(r['messages'], tokenize=False) for r in train_records[:cfg.calib_samples]]
if not calib_texts:
    calib_texts = ['Optimiza este prompt de ejemplo por favor.']

try:
    from gptqmodel import GPTQModel, QuantizeConfig
    qcfg = QuantizeConfig(bits=cfg.quant_bits, group_size=cfg.quant_group_size, desc_act=True)
    qmodel = GPTQModel.load(FULL_MODEL_DIR, qcfg)
    qmodel.quantize(calib_texts, batch_size=1)
    qmodel.save(cfg.quant_out_dir)
    tokenizer.save_pretrained(cfg.quant_out_dir)
    QUANT_METHOD = 'gptq-int4'
    print('Cuantizacion GPTQ INT4 completada en', cfg.quant_out_dir)
except Exception as e:
    print('GPTQ no disponible/fallo (', repr(e), '). Fallback a bitsandbytes NF4 4-bit.')
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    qmodel = AutoModelForCausalLM.from_pretrained(FULL_MODEL_DIR, quantization_config=bnb, device_map='auto')
    qmodel.save_pretrained(cfg.quant_out_dir)
    tokenizer.save_pretrained(cfg.quant_out_dir)
    QUANT_METHOD = 'bnb-nf4-4bit'
    print('Cuantizacion NF4 4-bit completada en', cfg.quant_out_dir)

## 10. Prueba del modelo cuantizado INT4

In [ ]:
gc.collect()
torch.cuda.empty_cache()

if QUANT_METHOD == 'gptq-int4':
    from gptqmodel import GPTQModel
    q = GPTQModel.load(cfg.quant_out_dir)
    q_infer = q.model if hasattr(q, 'model') else q
else:
    from transformers import AutoModelForCausalLM
    q_infer = AutoModelForCausalLM.from_pretrained(cfg.quant_out_dir, device_map='auto')

sample = 'Podrias, por favor, ayudarme a escribir un mensaje breve pero muy amable para felicitar a un companero por su reciente ascenso?'
print('ORIGINAL:', sample)
print('OPTIMIZADO (INT4):', optimize_prompt(q_infer, tokenizer, sample))

## 11. Exportacion de la carpeta lista para Docker

Genera una carpeta autocontenida con: pesos + config + tokenizer del modelo INT4, `metadata.json`, `requirements.txt`, `infer.py`, un `Dockerfile` de referencia y un `README.md`. Copiala a cualquier maquina o metela en una imagen Docker.

In [ ]:
import json, os, shutil, datetime
from pathlib import Path

src = cfg.quant_out_dir
dst = cfg.export_dir
Path(dst).mkdir(parents=True, exist_ok=True)

# 1) Copiar TODOS los archivos del modelo cuantizado (pesos, config.json, tokenizer...).
for name in os.listdir(src):
    s = os.path.join(src, name)
    if os.path.isfile(s):
        shutil.copy2(s, os.path.join(dst, name))

# 2) metadata.json con toda la informacion del modelo y el entrenamiento.
metadata = {
    'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'base_model': cfg.base_model_id,
    'task': 'prompt-optimization: menos tokens, sin redundancias, misma intencion',
    'instruction': INSTRUCTION,
    'finetuning': {
        'type': 'DoRA (Weight-Decomposed LoRA), fusionado en la base',
        'lora_r': cfg.lora_r,
        'lora_alpha': cfg.lora_alpha,
        'lora_dropout': cfg.lora_dropout,
        'target_modules': cfg.lora_target_modules,
        'use_dora': True,
        'epochs': cfg.epochs,
        'learning_rate': cfg.learning_rate,
        'lr_scheduler': cfg.lr_scheduler,
        'warmup_ratio': cfg.warmup_ratio,
        'weight_decay': cfg.weight_decay,
        'label_smoothing': cfg.label_smoothing,
        'neftune_noise_alpha': cfg.neftune_noise_alpha,
        'early_stopping_patience': cfg.early_stopping_patience,
        'max_seq_len': cfg.max_seq_len,
        'assistant_only_loss': bool(use_assistant_only),
    },
    'quantization': {'method': QUANT_METHOD, 'bits': cfg.quant_bits, 'group_size': cfg.quant_group_size},
    'hardware': {'trained_on': 'AMD Instinct MI300X', 'stack': 'ROCm 7.2.4 + PyTorch (Ubuntu 24.04)'},
}
with open(os.path.join(dst, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# 3) requirements.txt
reqs = ['transformers>=4.57.0', 'accelerate>=1.2.0', 'tokenizers>=0.20.0', 'safetensors>=0.4.5', 'torch>=2.11.0']
reqs.append('gptqmodel>=1.4.0' if QUANT_METHOD == 'gptq-int4' else 'bitsandbytes>=0.45.0')
with open(os.path.join(dst, 'requirements.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(reqs) + '\n')

# 4) infer.py (script de inferencia autonomo, usa 'dtype' no el deprecado 'torch_dtype').
infer_py = r'''import sys, os, json
from transformers import AutoTokenizer, AutoModelForCausalLM

HERE = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, 'metadata.json'), encoding='utf-8') as f:
    META = json.load(f)
INSTRUCTION = META['instruction']

tok = AutoTokenizer.from_pretrained(HERE)
model = AutoModelForCausalLM.from_pretrained(HERE, dtype='auto', device_map='auto')

def optimize(prompt):
    msgs = [{'role': 'user', 'content': INSTRUCTION + chr(10) + chr(10) + prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

if __name__ == '__main__':
    p = ' '.join(sys.argv[1:]) or 'Escribe aqui tu prompt largo y redundante.'
    print(optimize(p))
'''
with open(os.path.join(dst, 'infer.py'), 'w', encoding='utf-8') as f:
    f.write(infer_py)

# 5) Dockerfile de referencia (este pipeline NO construye la imagen).
dockerfile = r'''# Imagen de referencia para servir el modelo. Construir: docker build -t prompt-optimizer .
FROM rocm/pytorch:latest
WORKDIR /app
COPY . /app
RUN pip install --no-cache-dir -r requirements.txt
CMD python infer.py
'''
with open(os.path.join(dst, 'Dockerfile'), 'w', encoding='utf-8') as f:
    f.write(dockerfile)

# 6) README.md
readme = r'''# Gemma 4 E2B - Optimizador de Prompts (INT4)

Modelo experto (DoRA alto rango fusionado + INT4) cuyo unico proposito es optimizar prompts: menos tokens, sin redundancias, misma intencion.
Entrenado en AMD Instinct MI300X con ROCm 7.2.4 (Ubuntu 24.04).

## Uso local

    pip install -r requirements.txt
    python infer.py "tu prompt largo y redundante aqui"

## Docker

    docker build -t prompt-optimizer .
    docker run --rm --device=/dev/kfd --device=/dev/dri prompt-optimizer

Consulta metadata.json para los detalles del entrenamiento y la cuantizacion.
'''
with open(os.path.join(dst, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme)

print('Export listo en:', dst)
print('Archivos:', sorted(os.listdir(dst)))

## Resumen

Al terminar tendras:

- `outputs/gemma4-e2b-prompt-optimizer-dora/`: adapter DoRA entrenado (mejor checkpoint por early stopping).
- `outputs/gemma4-e2b-prompt-optimizer-merged/`: modelo completo con el adapter DoRA fusionado en la base (bf16).
- `outputs/gemma4-e2b-prompt-optimizer-int4/`: modelo cuantizado INT4.
- `export/gemma4-e2b-prompt-optimizer/`: carpeta autocontenida (pesos + config + tokenizer + `metadata.json` + `infer.py` + `requirements.txt` + `Dockerfile` + `README.md`) lista para copiar o incluir en una imagen Docker.

Claves para que aprenda y NO memorice: DoRA de alto rango con base congelada, dataset amplio y variado, early stopping sobre `eval_loss`, weight decay + lora_dropout + label smoothing, y vigilar la brecha train/eval del diagnostico.